In [ ]:
import pandas as pd
import os

df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/poster_image_scores.csv")

df['poster_title'] = df['image_path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0].replace('_', ' '))

print(df[['image_path', 'poster_title']].head())

                                          image_path  \
0           poster_images/poster_images/Superman.jpg   
1  poster_images/poster_images/The_Conjuring_Last...   
2              poster_images/poster_images/Ligaw.jpg   
3  poster_images/poster_images/El_Cas_Àngelus_La_...   
4  poster_images/poster_images/Jurassic_World_Reb...   

                           poster_title  
0                              Superman  
1              The Conjuring Last Rites  
2                                 Ligaw  
3  El Cas Àngelus La fascinació de Dalí  
4                Jurassic World Rebirth  


In [2]:
face_matches_df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/face_matches.csv")

In [3]:
df[df.image_path == "poster_images/poster_images/_Tis_the_Season_for_Love_2015_.jpg"]

,image_path,imdb_score,score_range,poster_title
29871,poster_images/poster_images/_Tis_the_Season_fo...,6.7,6–7,Tis the Season for Love 2015


In [4]:
df.to_csv("tesnime.csv",index=False)

In [5]:
df.shape

(49143, 4)

In [6]:
image_with_actors_df = pd.read_csv("/kaggle/input/computer-vision-project-dataset/poster_image_scores_with_actors.csv")

In [7]:
image_with_actors_df = image_with_actors_df[["image_path","popularity"]]


In [ ]:
combined_df = pd.merge(
    image_with_actors_df,
    df,
    on='image_path',
    how='right'  
)

print(combined_df.head())
print(f"Combined row count: {len(combined_df)}")

                                          image_path  popularity  imdb_score  \
0           poster_images/poster_images/Superman.jpg        15.0         0.0   
1  poster_images/poster_images/The_Conjuring_Last...        15.0         0.0   
2              poster_images/poster_images/Ligaw.jpg        15.0         0.0   
3  poster_images/poster_images/El_Cas_Àngelus_La_...        15.0         0.0   
4  poster_images/poster_images/Jurassic_World_Reb...        15.0         0.0   

  score_range                          poster_title  
0         0–1                              Superman  
1         0–1              The Conjuring Last Rites  
2         0–1                                 Ligaw  
3         0–1  El Cas Àngelus La fascinació de Dalí  
4         0–1                Jurassic World Rebirth  
Combined row count: 49311


In [9]:
bert_embeddings = pd.read_csv("/kaggle/input/embeddings-1/titles_with_bert_embeddings_1.csv")

In [ ]:
bert_embeddings['bert_cls_embedding'] = bert_embeddings['bert_cls_embedding'].apply(
    lambda x: [float(i) for i in x.split(',')] if isinstance(x, str) else x
)

print(len(bert_embeddings['bert_cls_embedding'].iloc[0]))
print(type(bert_embeddings['bert_cls_embedding'].iloc[0]))  

768
<class 'list'>


In [11]:
combined_df.shape

(49311, 5)

In [12]:
bert_embeddings= bert_embeddings[["image_path","bert_cls_embedding"]]

In [ ]:
final_df = pd.merge(
    combined_df,
    bert_embeddings,
    on='image_path',
    how='right'  
)

print(final_df.head())
print(f"Final row count: {len(final_df)}")

                                          image_path  popularity  imdb_score  \
0           poster_images/poster_images/Superman.jpg        15.0         0.0   
1  poster_images/poster_images/The_Conjuring_Last...        15.0         0.0   
2              poster_images/poster_images/Ligaw.jpg        15.0         0.0   
3  poster_images/poster_images/El_Cas_Àngelus_La_...        15.0         0.0   
4  poster_images/poster_images/Jurassic_World_Reb...        15.0         0.0   

  score_range                          poster_title  \
0         0–1                              Superman   
1         0–1              The Conjuring Last Rites   
2         0–1                                 Ligaw   
3         0–1  El Cas Àngelus La fascinació de Dalí   
4         0–1                Jurassic World Rebirth   

                                  bert_cls_embedding  
0  [-0.86040306, -0.120517045, -0.83552206, 0.296...  
1  [-1.0534014, -0.4176956, -0.37494773, -0.19581...  
2  [-0.4294514, -0.1105

In [14]:
final_df.rename(columns={'popularity': 'actor_score'}, inplace=True)
final_df.rename(columns={'bert_cls_embedding': 'title_embedding'}, inplace=True)

In [ ]:
# Import necessary libraries
import os, random, time
import pandas as pd
from PIL import Image, UnidentifiedImageError, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import lightning as L
from torchmetrics.regression import MeanAbsoluteError, MeanSquaredError

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Configuration parameters
POSTER_DIR = "/kaggle/input/computer-vision-project-dataset/poster_images/"
IMG_COL    = "image_path"
SCORE_COL  = "imdb_score"
EPOCHS     = 10
BATCH_SIZE = 128
LR         = 1e-4

# Define a custom dataset class for poster images
class PosterOnlyDS(Dataset):
    def __init__(self, df, root, tfm):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.tfm = tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root, row[IMG_COL])
        try:
            img = Image.open(path).convert("RGB")
        except (FileNotFoundError, UnidentifiedImageError):
            # Handle missing or invalid images by returning the next sample
            return self.__getitem__((idx + 1) % len(self))
        x_img = self.tfm(img)
        y = torch.tensor(row[SCORE_COL], dtype=torch.float32)
        return x_img, y

# Define a Vision Transformer (ViT) model for regression
class ViTOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model("vit_base_patch16_224", pretrained=True)
        self.vit.head = nn.Linear(self.vit.head.in_features, 1)

    def forward(self, x_img):
        return self.vit(x_img).squeeze(1)

# Define a LightningModule for training and validation
class Regr(L.LightningModule):
    def __init__(self, model, lr=LR):
        super().__init__()
        self.model = model
        self.lr = lr
        self.mae = MeanAbsoluteError()
        self.mse = MeanSquaredError()

    def forward(self, x_img):
        return self.model(x_img)

    def _step(self, batch, tag):
        x_img, y = batch
        yhat = self(x_img)
        loss = nn.MSELoss()(yhat, y)
        self.log(f"{tag}_mae", self.mae(yhat, y), prog_bar=True, batch_size=len(y))
        self.log(f"{tag}_mse", self.mse(yhat, y), prog_bar=True, batch_size=len(y))
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

# Function to build data loaders for training and validation
def build_loaders(df, img_size=224, bs=32, workers=2):
    # Filter out rows with missing image files
    df = df[df[IMG_COL].apply(lambda p: os.path.exists(os.path.join(POSTER_DIR, p)))]
    val_df = df.sample(frac=0.1, random_state=SEED)
    tr_df  = df.drop(val_df.index)

    # Define image transformations
    mean, std = [0.5]*3, [0.5]*3
    tr_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(.2, .2, .2, .1),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    val_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), transforms.Normalize(mean, std)])

    # Create data loaders
    tr_loader = DataLoader(
        PosterOnlyDS(tr_df, POSTER_DIR, tr_tfm),
        batch_size=bs, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader = DataLoader(
        PosterOnlyDS(val_df, POSTER_DIR, val_tfm),
        batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
    return tr_loader, val_loader

# Function to train the model
def train_one(df, epochs=EPOCHS, bs=BATCH_SIZE, freeze_vit=False):
    tr_loader, val_loader = build_loaders(df, bs=bs)
    model = ViTOnly()

    # Optionally freeze the ViT backbone
    if freeze_vit:
        for param in model.vit.parameters():
            param.requires_grad = False
        print("ViT backbone is frozen.")
    else:
        print("ViT backbone is trainable.")

    lit_model = Regr(model)

    # Define a checkpoint callback to save the best model
    checkpoint_cb = L.pytorch.callbacks.ModelCheckpoint(
        monitor="val_mae", mode="min", save_top_k=1,
        filename="best-vit-only-{epoch:02d}-{val_mae:.3f}"
    )

    print(f"Training on {len(tr_loader.dataset):,} samples")
    print(f"Validating on {len(val_loader.dataset):,} samples")

    # Initialize the Lightning trainer
    trainer = L.Trainer(
        max_epochs=epochs,
        precision="16-mixed" if torch.cuda.is_available() else 32,
        accelerator="auto",
        deterministic=True,
        log_every_n_steps=10,
        callbacks=[checkpoint_cb]
    )

    # Train the model
    t0 = time.time()
    trainer.fit(lit_model, tr_loader, val_loader)
    metrics = trainer.validate(lit_model, val_loader, verbose=False)[0]
    print(f"\nFinal MAE={metrics['val_mae']:.3f}, MSE={metrics['val_mse']:.3f}, Time={(time.time() - t0)/60:.1f} min")
    print(f"Best checkpoint saved to: {checkpoint_cb.best_model_path}")

    # Display sample predictions
    lit_model.eval()
    with torch.no_grad():
        for x_img, y_true in val_loader:
            y_pred = lit_model(x_img.to(lit_model.device))
            print("\nSample predictions:")
            for i in range(min(5, len(y_true))):
                print(f"True: {y_true[i].item():.2f}, Predicted: {y_pred[i].item():.2f}")
            break

    return metrics

# Run the training process
metrics = train_one(final_df)

# Print final metrics
print("\nFinal Metrics:")
print("MAE:", metrics["val_mae"], "MSE:", metrics["val_mse"])


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


🔥 ViT backbone is trainable.
🧪 Training on 44,380 samples
🧾 Validating on 4,931 samples


2025-06-10 17:05:40.689861: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749575140.887386      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749575140.945533      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type              | Params | Mode 
----------------------------------------------------
0 | model | ViTOnly           | 85.8 M | train
1 | mae   | MeanAbsoluteError | 0      | train
2 | mse   | MeanSquaredError  | 0      | train
----------------------------------------------------
85.8 M    Trainable params
0         Non-trainable params
85.8 M    Total params
343.198

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]


▶ FINAL MAE=1.728  MSE=5.932  |  128.8 min
📦 Best checkpoint saved to: /kaggle/working/lightning_logs/version_0/checkpoints/best-vit-only-epoch=01-val_mae=1.672.ckpt

🔍 Sample predictions:
🎯 True: 2.50 → 🧠 Pred: 3.23
🎯 True: 5.60 → 🧠 Pred: 7.07
🎯 True: 6.60 → 🧠 Pred: 6.71
🎯 True: 7.10 → 🧠 Pred: 6.45
🎯 True: 0.00 → 🧠 Pred: -0.48

Final Metrics:
MAE: 1.727739930152893 MSE: 5.932132720947266
